# Introduction to rosaceAA (Python)

This vignette replicates the [rosaceAA R vignette](https://github.com/pimentellab/rosace-aa/blob/main/vignettes/intro_rosaceAA.Rmd) in Python using the `rosace` package.

rosaceAA extends the base ROSACE model with position-level shrinkage and BLOSUM62-based amino acid grouping,
enabling prior information about substitution biochemistry to inform variant effect estimates.

## Four ROSACE models

| Model | Extra structure | Key Stan data |
|-------|-----------------|---------------|
| **ROSACE0** | None — independent variant scores | — |
| **ROSACE1** | Position-level grouping | `vMAPp`, `P` |
| **ROSACE2** | Position + global BLOSUM62 group | `vMAPp`, `P`, `vMAPb`, `B` |
| **ROSACE3** | Position + BLOSUM62 + position-specific activation | same as ROSACE2 + `rho` |

We use the **OCT1 drug cytotoxicity screen** throughout, mirroring the R vignette.


## Installation

```bash
pip install rosace
```

CmdStan (required for Bayesian MCMC inference) can be installed via:
```python
import cmdstanpy
cmdstanpy.install_cmdstan()
```


In [ ]:
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

import os
import re
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rdata

from rosace.assay import AssayGrowth, AssaySetGrowth
from rosace.preprocessing import filter_data, impute_data, normalize_data
from rosace.run_rosace import run_rosace
from rosace.utils import output_score, map_blosum_score
from rosace.visualization import score_density


## Install CmdStan

The cell below installs CmdStan if it is not already present.  Skip it if you have already installed CmdStan.


In [ ]:
import cmdstanpy

# Issue with Homebrew clang on macOS:
os.environ.setdefault("CC", "/usr/bin/clang")
os.environ.setdefault("CXX", "/usr/bin/clang++")

cmdstanpy.install_cmdstan(overwrite=False, verbose=True)


## Read count data

The OCT1 drug cytotoxicity screen has **three replicates**.  Each replicate is stored as a DataFrame in `oct1.rda`.

| Object | Columns | Timepoints |
|--------|---------|------------|
| `oct1_rep1` | hgvs, c_0, c_2, c_4, c_6 | 4 |
| `oct1_rep2` | hgvs, c_0, c_2, c_4, c_6 | 4 |
| `oct1_rep3` | hgvs, c_0, c_2, c_4 | 3 |


In [ ]:
# Locate the .rda file relative to this notebook
notebook_dir = Path(".").resolve()
repo_root = notebook_dir.parent
rda_src = repo_root / "data" / "R" / "oct1.rda"

# Copy to a temporary directory so we work from a clean location
tmp_dir = Path(tempfile.mkdtemp())
rda_path = shutil.copy(str(rda_src), str(tmp_dir))
print(f"Working copy: {rda_path}")

# Read all objects from the .rda file
oct1_data = rdata.read_rda(rda_path)
print(f"Objects in .rda: {list(oct1_data.keys())}")

key = "OCT1"
type_ = "growth"


## Create Assay objects

Each replicate becomes an `AssayGrowth`.  The `key` ties replicates together (mirrors the R `key = "1SM73"`).


In [ ]:
def make_assay(df_raw: pd.DataFrame, rep: int) -> AssayGrowth:
    """Convert a replicate DataFrame from oct1.rda into an AssayGrowth."""
    df = df_raw.copy()
    df.columns = [str(c) for c in df.columns]
    count_cols = [c for c in df.columns if c.startswith("c_")]
    return AssayGrowth(
        counts=df[count_cols].values.astype(float),
        var_names=df["hgvs"].tolist(),
        key=key,
        rep=rep,
    )


assay1 = make_assay(oct1_data["oct1_rep1"], rep=1)
assay2 = make_assay(oct1_data["oct1_rep2"], rep=2)
assay3 = make_assay(oct1_data["oct1_rep3"], rep=3)

for a in [assay1, assay2, assay3]:
    print(f"Replicate {a.rep}: {a.counts.shape[0]} variants × {a.counts.shape[1]} timepoints")


## Preprocessing

### Filter
`filter_data` removes variants with too many NAs or too few total counts (mirrors R `FilterData`).


In [ ]:
filtered_assays = []
for a in [assay1, assay2, assay3]:
    fa = filter_data(a, na_rmax=0.5, min_count=20)
    print(f"Rep {a.rep}: {a.counts.shape[0]} → {fa.counts.shape[0]} variants after filtering")
    filtered_assays.append(fa)


### Impute

`impute_data` fills remaining NAs.  We use K-nearest neighbours (KNN) imputation to match the R vignette (`ImputeData(impute.method = "knn")`).


In [ ]:
# KNN imputation mirrors the R vignette default
imputed_assays = [impute_data(a, method="knn") for a in filtered_assays]
print("KNN imputation complete.")


### Normalize

`normalize_data` computes log-ratio growth scores relative to the wildtype controls (mirrors R `NormalizeData(normalization.method = "wt")`).


In [ ]:
norm_assays = []
for a in imputed_assays:
    na = normalize_data(a, method="wt", wt_var_names=["_wt"], wt_rm=True)
    print(f"Rep {a.rep}: {na.norm_counts.shape[0]} variants after wt-normalisation")
    norm_assays.append(na)


### Integrate replicates

Combine the three normalised replicates into an `AssaySetGrowth` via outer join (mirrors R `IntegrateData`).


In [ ]:
def integrate_replicates(assays: list[AssayGrowth]) -> AssaySetGrowth:
    """Integrate any number of normalized AssayGrowth objects via outer join."""

    def norm_df(a: AssayGrowth) -> pd.DataFrame:
        names = a.norm_var_names or a.var_names
        T = a.norm_counts.shape[1]
        return pd.DataFrame(
            a.norm_counts, index=names,
            columns=[f"r{a.rep}_t{t}" for t in range(T)],
        )

    def raw_df(a: AssayGrowth) -> pd.DataFrame:
        T = a.counts.shape[1]
        return pd.DataFrame(
            a.counts, index=a.var_names,
            columns=[f"r{a.rep}_raw_t{t}" for t in range(T)],
        )

    combined = norm_df(assays[0])
    for a in assays[1:]:
        combined = combined.join(norm_df(a), how="outer")

    raw_combined = raw_df(assays[0])
    for a in assays[1:]:
        raw_combined = raw_combined.join(raw_df(a), how="outer")

    return AssaySetGrowth(
        combined_counts=combined.values,
        var_names=list(combined.index),
        reps=[a.rep for a in assays],
        key=assays[0].key,
        raw_counts=raw_combined.values,
        rounds=[a.rounds for a in assays],
    )


rosace_set = integrate_replicates(norm_assays)
print(
    f"AssaySetGrowth: {rosace_set.combined_counts.shape[0]} variants, "
    f"{rosace_set.combined_counts.shape[1]} columns "
    f"(reps {rosace_set.reps}, rounds per rep: {rosace_set.rounds})"
)


## Process variant metadata

Parse HGVS-style names like `p.(A107C)` into `position`, `wildtype`, `mutation`, and `type` (mirrors R `rosace@var.data` manipulation in the vignette).


In [ ]:
_HGVS_RE = re.compile(r"p\.\(([A-Z])(\d+)([A-Z])\)")


def parse_hgvs(hgvs: str) -> tuple:
    """Return (wildtype, position, mutation, type) from an HGVS protein name."""
    m = _HGVS_RE.match(hgvs)
    if m:
        wt, pos, mut = m.group(1), int(m.group(2)), m.group(3)
        vtype = "synonymous" if wt == mut else "missense"
        return wt, pos, mut, vtype
    if "del" in hgvs:
        return None, None, None, "deletion"
    return None, None, None, "other"


parsed = [parse_hgvs(v) for v in rosace_set.var_names]
var_data = pd.DataFrame(
    parsed,
    columns=["wildtype", "position", "mutation", "type"],
    index=rosace_set.var_names,
).rename_axis("variants").reset_index()

print("Variant type distribution:")
print(var_data["type"].value_counts())
var_data.head()


## Subset for vignette

For this vignette we run on the first **1 000 variants** of the integrated assay set — matching the structure of the R vignette (which uses 100).
Remove the slicing to run the full dataset (may take an hour).


In [ ]:
# ---- ROSACE0 test set: first 1000 of all variant types ----
N = 1000
all_idx = list(range(min(N, len(rosace_set.var_names))))
assay_test0 = AssaySetGrowth(
    combined_counts=rosace_set.combined_counts[all_idx],
    var_names=[rosace_set.var_names[i] for i in all_idx],
    reps=rosace_set.reps,
    key=rosace_set.key,
    raw_counts=rosace_set.raw_counts[all_idx] if rosace_set.raw_counts is not None else None,
    rounds=rosace_set.rounds,
)

# ---- ROSACE1/2/3 test set: first 1000 missense variants ----
var_info_full = (
    var_data[var_data["type"] == "missense"]
    .rename(columns={"position": "pos", "variants": "variant"})
    [["variant", "pos", "wildtype", "mutation"]]
    .rename(columns={"wildtype": "wt", "mutation": "mut"})
    .copy()
)
var_info_full["pos"] = var_info_full["pos"].astype(int)

missense_set = set(var_info_full["variant"])
missense_idx = [i for i, v in enumerate(rosace_set.var_names) if v in missense_set]

assay_missense = AssaySetGrowth(
    combined_counts=rosace_set.combined_counts[missense_idx],
    var_names=[rosace_set.var_names[i] for i in missense_idx],
    reps=rosace_set.reps,
    key=rosace_set.key,
    raw_counts=rosace_set.raw_counts[missense_idx] if rosace_set.raw_counts is not None else None,
    rounds=rosace_set.rounds,
)

keep_vars = assay_missense.var_names[:N]
keep_set = set(keep_vars)
var_info_test = var_info_full[var_info_full["variant"].isin(keep_set)].copy()
m_idx = [i for i, v in enumerate(assay_missense.var_names) if v in keep_set]
assay_test = AssaySetGrowth(
    combined_counts=assay_missense.combined_counts[m_idx],
    var_names=[assay_missense.var_names[i] for i in m_idx],
    reps=assay_missense.reps,
    key=assay_missense.key,
    raw_counts=assay_missense.raw_counts[m_idx] if assay_missense.raw_counts is not None else None,
    rounds=assay_missense.rounds,
)

print(f"ROSACE0 assay:     {assay_test0.combined_counts.shape[0]} variants")
print(f"ROSACE1/2/3 assay: {assay_test.combined_counts.shape[0]} missense variants")
print(f"var_info_test:     {len(var_info_test)} rows, {var_info_test['pos'].nunique()} positions")


## Run ROSACE

There are **four model variants**, matching the `rosaceAA` R package. All are fitted with `run_rosace()`.

### Model 0 — no positional structure

The simplest model: each variant gets its own fitness score with no position- or amino-acid-level pooling (equivalent to `RunRosace` with no `pos.col` in R).


In [ ]:
# Model 0 – run on AssaySetGrowth with all variant types
score_model0 = run_rosace(
    assay_test0,
    method="ROSACE0",
    seed=42,
)
print(f"ROSACE0  |  variants: {len(score_model0.score)}")
score_model0.score.head()


### Model 1 — position-level grouping

A position-level hyperprior pools information across all variants at the same amino-acid position.  Equivalent to `RunRosace(pos.col = "position")` in R.

```
beta[v] = phi[vMAPp[v]] + eta2[v] * sqrt(sigma2[vMAPp[v]])
m[v,t]  ~ N(b[v] + beta[v]*t,  sqrt(epsilon2[vMAPm[v]]))
```


In [ ]:
# Model 1 – requires var_info (missense variants only)
score_model1 = run_rosace(
    assay_test,
    method="ROSACE1",
    var_info=var_info_test,
    seed=42,
)
print(f"ROSACE1  |  variants: {len(score_model1.score)}, "
      f"positions: {len(score_model1.optional_score)}")
score_model1.score.head()


### Model 2 — position + global BLOSUM62 group (rosaceAA)

Adds a **global** BLOSUM62 substitution-score group (`vMAPb`) on top of position shrinkage. Substitutions with similar BLOSUM62 scores share a hyperprior, encoding the biochemical expectation that similar substitutions have similar effects. Equivalent to `RunRosace(wt.col, mut.col, aa.code = "single", pos.act = FALSE)` in R.

```
beta[v] = phi[p] + psi[b] + eta2[v]*sqrt(sigma2[p]) + eta3[v]*sqrt(tau2[b])
```


In [ ]:
# Model 2 – rosaceAA: position + BLOSUM62 group
score_model2 = run_rosace(
    assay_test,
    method="ROSACE2",
    var_info=var_info_test,
    seed=42,
)
print(f"ROSACE2  |  variants: {len(score_model2.score)}, "
      f"positions: {len(score_model2.optional_score)}")
score_model2.score.head()


### Model 3 — position + BLOSUM62 + activation (rosaceAA)

Adds a global activation fraction `rho ∈ (0,1)` that scales the entire signal. Equivalent to `RunRosace(..., pos.act = TRUE)` in R.

```
beta[v] = rho * (phi[p] + psi[b] + eta2[v]*sqrt(sigma2[p]) + eta3[v]*sqrt(tau2[b]))
rho ~ Beta(2, 2)
```


In [ ]:
# Model 3 – rosaceAA: position + BLOSUM62 + activation fraction
score_model3 = run_rosace(
    assay_test,
    method="ROSACE3",
    var_info=var_info_test,
    seed=42,
)
print(f"ROSACE3  |  variants: {len(score_model3.score)}, "
      f"positions: {len(score_model3.optional_score)}")
score_model3.score.head()


## Output scores

The `output_score` function extracts the **variant-level** scores from a `Score` object and adds hypothesis-test labels using the local false sign rate (LFSR). This mirrors R's `OutputScore` function.

| LFSR | Label |
|------|-------|
| ≥ 0.05 | Neutral |
| < 0.05, mean > 0 | Pos |
| < 0.05, mean ≤ 0 | Neg |

For **ROSACE1/2/3**, `score.optional_score` contains the position-level `phi` scores, equivalent to R's `OutputScore(pos.info = TRUE)$df_position`.


In [ ]:
# --- Variant-level scores (all models) ---
scores_model0 = output_score(score_model0, sig_test=0.05)
scores_model1 = output_score(score_model1, sig_test=0.05)
scores_model2 = output_score(score_model2, sig_test=0.05)
scores_model3 = output_score(score_model3, sig_test=0.05)

for name, sc in [("ROSACE0", scores_model0), ("ROSACE1", scores_model1),
                  ("ROSACE2", scores_model2), ("ROSACE3", scores_model3)]:
    cnts = sc["label"].value_counts().to_dict()
    print(f"{name}: Neg={cnts.get('Neg',0):4d}  Neutral={cnts.get('Neutral',0):4d}  "
          f"Pos={cnts.get('Pos',0):4d}")


In [ ]:
# ROSACE0 variant scores
print("ROSACE0 variant scores (head):")
display(scores_model0.head())


In [ ]:
# ROSACE1 variant scores + position-level scores
# Equivalent to OutputScore(pos.info = TRUE) in R
df_variant_model1 = scores_model1.copy()
df_position_model1 = score_model1.optional_score.copy()

print("ROSACE1 variant scores (head):")
display(df_variant_model1.head())

print("\nROSACE1 position scores (head):")
display(df_position_model1.head())


In [ ]:
# ROSACE2 variant scores + position-level scores
# Equivalent to OutputScore(pos.info = TRUE, blosum.info = TRUE) in R
# Add BLOSUM62 group annotation to variant scores (mirrors R's blosum.info = TRUE)
df_variant_model2 = scores_model2.copy()
df_variant_model2 = df_variant_model2.merge(
    var_info_test[["variant", "pos", "wt", "mut"]], on="variant", how="left"
)
df_variant_model2["blosum_group"] = df_variant_model2.apply(
    lambda r: map_blosum_score(str(r["wt"]), str(r["mut"]))
              if pd.notna(r["wt"]) and pd.notna(r["mut"]) else None,
    axis=1,
)
df_position_model2 = score_model2.optional_score.copy()

print("ROSACE2 variant scores with BLOSUM group (head):")
display(df_variant_model2.head())

print("\nROSACE2 position scores (head):")
display(df_position_model2.head())

# BLOSUM62 group-level scores (psi) – from misc (mirrors R's OutputScore internals)
if "blosum_scores" in score_model2.misc:
    print("\nROSACE2 BLOSUM62 group scores (psi):")
    display(score_model2.misc["blosum_scores"])


In [ ]:
# ROSACE3 variant scores + position-level scores
# Equivalent to OutputScore(pos.info = TRUE, blosum.info = TRUE, pos.act.info = TRUE) in R
df_variant_model3 = scores_model3.copy()
df_variant_model3 = df_variant_model3.merge(
    var_info_test[["variant", "pos", "wt", "mut"]], on="variant", how="left"
)
df_variant_model3["blosum_group"] = df_variant_model3.apply(
    lambda r: map_blosum_score(str(r["wt"]), str(r["mut"]))
              if pd.notna(r["wt"]) and pd.notna(r["mut"]) else None,
    axis=1,
)
df_position_model3 = score_model3.optional_score.copy()

print("ROSACE3 variant scores with BLOSUM group (head):")
display(df_variant_model3.head())

print("\nROSACE3 position scores (head):")
display(df_position_model3.head())

# BLOSUM62 group-level scores (psi)
if "blosum_scores" in score_model3.misc:
    print("\nROSACE3 BLOSUM62 group scores (psi):")
    display(score_model3.misc["blosum_scores"])

# Global activation fraction rho (pos.act.info in R vignette)
# In the Python Stan model, rho is a global scalar (not position-specific)
if "rho_mean" in score_model3.misc:
    print(f"\nROSACE3 global activation fraction:"
          f" rho = {score_model3.misc['rho_mean']:.4f} ± {score_model3.misc['rho_sd']:.4f}")


## Score comparison across models

Compare the four model scores on the missense variants to understand how increasing structural complexity affects estimates.


In [ ]:
# Build a merged comparison on the missense variants (common to models 1/2/3)
compare = (
    scores_model1[["variant", "mean"]].rename(columns={"mean": "ROSACE1"})
    .merge(scores_model2[["variant", "mean"]].rename(columns={"mean": "ROSACE2"}),
           on="variant")
    .merge(scores_model3[["variant", "mean"]].rename(columns={"mean": "ROSACE3"}),
           on="variant")
)

r12 = compare[["ROSACE1", "ROSACE2"]].corr().iloc[0, 1]
r13 = compare[["ROSACE1", "ROSACE3"]].corr().iloc[0, 1]
r23 = compare[["ROSACE2", "ROSACE3"]].corr().iloc[0, 1]
print(f"Pearson r  ROSACE1 vs ROSACE2 : {r12:.4f}")
print(f"Pearson r  ROSACE1 vs ROSACE3 : {r13:.4f}")
print(f"Pearson r  ROSACE2 vs ROSACE3 : {r23:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [("ROSACE1", "ROSACE2", r12), ("ROSACE1", "ROSACE3", r13), ("ROSACE2", "ROSACE3", r23)]
for ax, (xc, yc, r) in zip(axes, pairs):
    ax.scatter(compare[xc], compare[yc], alpha=0.15, s=5)
    lim = [min(compare[xc].min(), compare[yc].min()),
           max(compare[xc].max(), compare[yc].max())]
    ax.plot(lim, lim, "r--", linewidth=0.8)
    ax.set_xlabel(xc)
    ax.set_ylabel(yc)
    ax.set_title(f"r = {r:.4f}")
fig.suptitle("Score comparison across ROSACE models – OCT1 DMS", fontsize=12)
plt.tight_layout()
plt.show()


## Position-level effects

For ROSACE1/2/3, `score.optional_score` contains the position-level mean effect `phi[p]` — equivalent to R's `OutputScore(pos.info = TRUE)$df_position`. Positions with strongly negative phi are likely functionally critical.


In [ ]:
pos_scores = score_model1.optional_score.copy()
print(f"Position-level scores: {len(pos_scores)} positions")

print("\nTop 10 most negatively selected positions (phi):")
display(pos_scores.nsmallest(10, "mean").reset_index(drop=True))

print("\nTop 10 most positively selected positions (phi):")
display(pos_scores.nlargest(10, "mean").reset_index(drop=True))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(
    pos_scores["pos"], pos_scores["mean"],
    color=["steelblue" if m >= 0 else "tomato" for m in pos_scores["mean"]],
    width=0.8, alpha=0.8,
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Position")
ax.set_ylabel("phi (position effect)")
ax.set_title("ROSACE1 Position-Level Effects – OCT1 DMS")
plt.tight_layout()
plt.show()


## Top loss-of-function and gain-of-function variants

Use ROSACE1 scores (ROSACE2/3 are similar) to identify the most strongly deleterious and beneficial mutations.


In [ ]:
scores_meta = scores_model1.merge(
    var_data.rename(columns={"variants": "variant"}), on="variant", how="left"
)
cols = ["variant", "position", "wildtype", "mutation", "mean", "sd", "lfsr", "label"]

print("Top 10 Loss-of-Function variants (most negative ROSACE1 score):")
display(scores_meta.nsmallest(10, "mean")[cols].reset_index(drop=True))

print("\nTop 10 Gain-of-Function variants (most positive ROSACE1 score):")
display(scores_meta.nlargest(10, "mean")[cols].reset_index(drop=True))


## Score distribution

Density plot of ROSACE1 scores for the missense variants.


In [ ]:
fig = score_density(
    data=scores_meta,
    type_col="type",
    score_col="mean",
    order=["missense"],
    title="ROSACE1 Score Distribution – OCT1 DMS",
    show=False,
)
plt.tight_layout()
plt.show()


## Summary

This notebook demonstrated the full rosaceAA workflow on the **OCT1 DMS dataset**, replicating the [R vignette](https://github.com/pimentellab/rosace-aa/blob/main/vignettes/intro_rosaceAA.Rmd):

1. **Data loading** – `oct1.rda` read with `rdata`, three replicates converted to `AssayGrowth` objects.
2. **QC** – per-replicate filtering (`na_rmax=0.5, min_count=20`), KNN imputation, wildtype normalisation.
3. **Integration** – three replicates merged into a single `AssaySetGrowth` via outer join.
4. **Variant metadata** – HGVS names parsed into `position`, `wildtype`, `mutation`, `type`.
5. **Four ROSACE models** run via `run_rosace(method=...)`:
   - **ROSACE0** – baseline, no structural prior.
   - **ROSACE1** – position-level hierarchical shrinkage.
   - **ROSACE2** – position + global BLOSUM62 group (rosaceAA).
   - **ROSACE3** – ROSACE2 + global activation fraction `rho`.
6. **Score output** – variant-level `output_score()` and position-level `optional_score`.
7. **Visualisation** – cross-model correlation, position bar plots, score density.

**Python vs R differences:**

| Feature | R (`rosaceAA`) | Python (`rosace`) |
|---------|---------------|-------------------|
| Run on single replicate | `type = "Assay"` | Pass `AssayGrowth` to `run_rosace` |
| Run on all replicates | `type = "AssaySet"` | Pass `AssaySetGrowth` to `run_rosace` |
| `ctrl.col` / `stop.col` | Supported | Use `var_info` filtering instead |
| `OutputScore(blosum.info=TRUE)` | Returns BLOSUM group column | Derive from `var_info` + `map_blosum_score` |
| `OutputScore(pos.act.info=TRUE)` | Returns activation info | `rho` is a global scalar in the Stan model |
